# AI-Text Humanizer - Detector and T5 Modeling

## Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

## Load Dataset

In [ ]:
data_dir = 'Rajarshi-Roy-research___defactify_text_dataset/default/0.0.0/f510e7ced81c9af0aecc4648f5fe470b3c0132c7'
split_files = {
    'train': 'defactify_text_dataset-train.arrow',
    'validation': 'defactify_text_dataset-validation.arrow',
    'test': 'defactify_text_dataset-test.arrow'
}

df_train = Dataset.from_file(os.path.join(data_dir, split_files['train'])).to_pandas()
df_val = Dataset.from_file(os.path.join(data_dir, split_files['validation'])).to_pandas()
df_test = Dataset.from_file(os.path.join(data_dir, split_files['test'])).to_pandas()

print(df_train.shape, df_val.shape, df_test.shape)
df_train.head()

## Dataset Overview

In [ ]:
print(df_train['Label_A'].value_counts())
print()
print(df_train['Label_B'].value_counts())

## Handle Missing Values and Duplicates

In [ ]:
df_train = df_train.dropna(subset=['Text']).drop_duplicates(subset=['Text']).reset_index(drop=True)
df_val = df_val.dropna(subset=['Text']).drop_duplicates(subset=['Text']).reset_index(drop=True)
df_test = df_test.dropna(subset=['Text']).drop_duplicates(subset=['Text']).reset_index(drop=True)

print(df_train.shape, df_val.shape, df_test.shape)

## Detector - TF-IDF + Logistic Regression

`Label_A`: 0 = human, 1 = AI-generated. This detector acts as the oracle used later to score the humanizer's output.

In [ ]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english')

X_train = vectorizer.fit_transform(df_train['Text'])
X_val = vectorizer.transform(df_val['Text'])

y_train = df_train['Label_A']
y_val = df_val['Label_A']

clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

## Evaluate Detector

In [ ]:
val_preds = clf.predict(X_val)
val_probs = clf.predict_proba(X_val)[:, 1]

print(classification_report(y_val, val_preds, target_names=['Human', 'AI']))
print('ROC-AUC:', roc_auc_score(y_val, val_probs))

cm = confusion_matrix(y_val, val_preds)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI'])
fig, ax = plt.subplots(figsize=(5, 5))
cm_display.plot(ax=ax)
plt.show()

## Detector Score Function

In [ ]:
def detector_score(texts):
    X = vectorizer.transform(texts)
    return clf.predict_proba(X)[:, 1]

sample_texts = df_val['Text'].iloc[:3].tolist()
print(detector_score(sample_texts))

## Build Humanizer Training Pairs

The dataset isn't row-paired, so AI and human texts are matched randomly as a working stand-in for real (AI, human) pairs.

In [ ]:
def build_pairs(df, n_pairs=5000, random_state=42):
    human_df = df[df['Label_A'] == 0].sample(n=n_pairs, random_state=random_state, replace=True).reset_index(drop=True)
    ai_df = df[df['Label_A'] == 1].sample(n=n_pairs, random_state=random_state, replace=True).reset_index(drop=True)

    pairs = pd.DataFrame({
        'input_text': 'humanize: ' + ai_df['Text'],
        'target_text': human_df['Text']
    })
    return pairs

train_pairs = build_pairs(df_train, n_pairs=5000)
val_pairs = build_pairs(df_val, n_pairs=1000)

train_pairs.head()

## Load T5 Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

T5_MODEL = 't5-small'
t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL).to(device)

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256

## Tokenize Pairs

In [ ]:
def tokenize_pairs(batch):
    model_inputs = t5_tokenizer(batch['input_text'], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = t5_tokenizer(batch['target_text'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_pairs_ds = Dataset.from_pandas(train_pairs)
val_pairs_ds = Dataset.from_pandas(val_pairs)

train_pairs_ds = train_pairs_ds.map(tokenize_pairs, batched=True, remove_columns=train_pairs.columns.tolist())
val_pairs_ds = val_pairs_ds.map(tokenize_pairs, batched=True, remove_columns=val_pairs.columns.tolist())

## Fine-Tune T5

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tokenizer, model=t5_model)

seq2seq_args = Seq2SeqTrainingArguments(
    output_dir='./humanizer_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=50,
    predict_with_generate=True
)

seq2seq_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=seq2seq_args,
    train_dataset=train_pairs_ds,
    eval_dataset=val_pairs_ds,
    data_collator=data_collator
)

seq2seq_trainer.train()

## Humanize Function

In [ ]:
def humanize(text, max_length=256):
    input_text = 'humanize: ' + text
    inputs = t5_tokenizer(input_text, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(device)
    outputs = t5_model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_ai_text = df_val[df_val['Label_A'] == 1]['Text'].iloc[0]
print('--- ORIGINAL (AI) ---')
print(sample_ai_text[:400])
print()
print('--- HUMANIZED ---')
print(humanize(sample_ai_text))

## Evaluate Humanizer with Detector

A good humanizer should lower the detector's P(AI) score on humanized text compared to the original AI text.

In [ ]:
eval_sample = df_val[df_val['Label_A'] == 1]['Text'].sample(20, random_state=42).tolist()

original_scores = detector_score(eval_sample)
humanized_texts = [humanize(t) for t in eval_sample]
humanized_scores = detector_score(humanized_texts)

results = pd.DataFrame({
    'original_ai_score': original_scores,
    'humanized_score': humanized_scores,
    'delta': humanized_scores - original_scores
})

print(results.describe())
results.head()

## Save Models

In [ ]:
os.makedirs('models/humanizer_t5', exist_ok=True)
os.makedirs('models/detector_tfidf', exist_ok=True)

t5_model.save_pretrained('models/humanizer_t5')
t5_tokenizer.save_pretrained('models/humanizer_t5')

joblib.dump(vectorizer, 'models/detector_tfidf/vectorizer.joblib')
joblib.dump(clf, 'models/detector_tfidf/classifier.joblib')

print('Saved humanizer model to models/humanizer_t5/')
print('Saved detector to models/detector_tfidf/')